# 06 — ML Deployment & Serving
## Practical notebook for ML Engineer / Applied AI Engineer interviews

Topics: training vs inference, model serialization, preprocessing consistency, batch vs online inference, FastAPI serving, input validation, health/readiness, latency/throughput, logging/monitoring, artifact versioning, CPU/GPU, ONNX, Docker, stateless scaling, async inference, canary/shadow deployment, rollback, and interview preparation.

Run the notebook top-to-bottom. API, Docker and ONNX sections provide optional code/templates for separate execution.

## 1. Training vs inference

**Training** learns model parameters from historical data. **Inference** uses the trained artifact to predict on new inputs.

```text
Training data → preprocessing → fit → evaluate → versioned artifact
New request → validate → same preprocessing → predict → response
```

Do not retrain for every prediction request. Train offline; load the approved artifact in the serving process.

In [14]:
import json
import time
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# NumPy 2.x on Apple Silicon (Accelerate BLAS) emits spurious
# "divide by zero / overflow / invalid value encountered in matmul" warnings
# even on clean, finite data. Results are correct, so silence only these.
warnings.filterwarnings("ignore", message=r".*encountered in matmul", category=RuntimeWarning)

RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Bundle preprocessing and estimator together.
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])
model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))

Accuracy: 0.9825
ROC-AUC: 0.9954


## 2. Serialize the complete pipeline

Use `joblib` commonly with scikit-learn/NumPy-heavy objects. Save the preprocessing and estimator together, not just the final classifier.

**Security:** pickle/joblib loading can execute code. Load artifacts only from trusted sources. Python/library versions can also affect compatibility.

In [15]:
import joblib

model_path = ARTIFACT_DIR / "breast_cancer_pipeline.joblib"
joblib.dump(model, model_path)

loaded_model = joblib.load(model_path)
loaded_pred = loaded_model.predict(X_test)

print("Artifact:", model_path.resolve())
print("Predictions identical:", np.array_equal(pred, loaded_pred))

Artifact: /Users/vivek-18890/Python/learn_python/ml/13-ml-fundamentals/artifacts/breast_cancer_pipeline.joblib
Predictions identical: True


## 3. Pickle vs Joblib vs ONNX

- **Pickle:** Python object serialization; do not load untrusted files.
- **Joblib:** common for scikit-learn models and NumPy-heavy objects; also unsafe for untrusted artifacts.
- **ONNX:** portable representation for supported models/operators; can run with ONNX Runtime outside the original Python model environment.

Serialization stores an artifact; it does not itself provide an API, monitoring or deployment process.

## 4. Training-serving consistency and schema validation

Serving must use compatible feature names, types, order and preprocessing. A saved scikit-learn `Pipeline` helps preserve transformations.

Feature schema checks prevent missing or unexpected fields from silently producing bad predictions.

In [16]:
expected_features = list(X_train.columns)

def validate_features(frame, expected):
    missing = sorted(set(expected) - set(frame.columns))
    extra = sorted(set(frame.columns) - set(expected))
    if missing or extra:
        raise ValueError(f"Schema mismatch: missing={missing}, extra={extra}")
    return frame.loc[:, expected]

sample = validate_features(X_test.iloc[[0]].copy(), expected_features)
print("Prediction:", int(loaded_model.predict(sample)[0]))

Prediction: 0


## 5. Batch vs online inference

**Online:** client → API → model → response. Use when a caller needs a prediction immediately.

**Batch:** data source → scheduled/queued job → predictions for many records → result store. Use when immediate response is unnecessary and throughput/cost matters more than per-request latency.

In [17]:
batch = X_test.iloc[:20]
start = time.perf_counter()
batch_pred = loaded_model.predict(batch)
batch_prob = loaded_model.predict_proba(batch)[:, 1]
elapsed_ms = (time.perf_counter() - start) * 1000

results = pd.DataFrame({
    "prediction": batch_pred,
    "probability_class_1": batch_prob
}, index=batch.index)
display(results.head())
print(f"20-row batch inference: {elapsed_ms:.2f} ms")

,prediction,probability_class_1
256,0,5.888242e-08
428,1,9.999887e-01
501,0,6.410825e-03
363,1,5.335085e-01
564,0,6.525001e-10


20-row batch inference: 1.35 ms


## 6. Latency, throughput, concurrency

- **Latency:** time for a request to complete.
- **Throughput:** requests/records processed per unit time.
- **Concurrency:** requests handled at once.
- **p95 latency:** 95% of measured requests finish at or below this time; p99 highlights the slow tail.

Local measurements are only a starting point; production performance depends on hardware, model warm-up, payload size, concurrency and network overhead.

In [18]:
sample = X_test.iloc[[0]]
loaded_model.predict(sample)  # warm-up

latencies_ms = []
for _ in range(100):
    start = time.perf_counter()
    loaded_model.predict(sample)
    latencies_ms.append((time.perf_counter() - start) * 1000)

for p in [50, 95, 99]:
    print(f"p{p} latency (ms): {np.percentile(latencies_ms, p):.3f}")

p50 latency (ms): 0.347
p95 latency (ms): 0.480
p99 latency (ms): 0.521


## 7. REST model serving with FastAPI

FastAPI can expose a trained model through HTTP. The service should validate inputs, return a stable response schema, load the model once at startup, handle errors safely and expose operational endpoints.

Install: `pip install fastapi uvicorn pandas scikit-learn joblib`

The next cell uses `%%writefile` to save the service as `app.py` next to the `artifacts/` directory. It is a separate program, so it is written to disk rather than executed inside the notebook.

In [19]:
%%writefile app.py
from pathlib import Path
import logging
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

MODEL_PATH = Path("artifacts/breast_cancer_pipeline.joblib")
MODEL_VERSION = "1.0.0"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ml-serving")
app = FastAPI(title="ML Prediction API", version=MODEL_VERSION)

# Load once when the process starts, not on every request.
# If loading fails, keep the process up but report "not ready" (503) instead of crashing.
try:
    model = joblib.load(MODEL_PATH)
    expected_features = list(model.feature_names_in_)
except Exception:
    logger.exception("Model failed to load")
    model, expected_features = None, []

class PredictionRequest(BaseModel):
    features: dict[str, float]

class PredictionResponse(BaseModel):
    prediction: int
    probability_class_1: float
    model_version: str

@app.get("/health")
def health():
    # Liveness: the process can respond.
    return {"status": "ok"}

@app.get("/ready")
def ready():
    # Readiness: model is loaded and the service can serve.
    if model is None:
        raise HTTPException(status_code=503, detail="Model not ready")
    return {"status": "ready", "model_version": MODEL_VERSION}

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not ready")
    missing = set(expected_features) - set(request.features)
    extra = set(request.features) - set(expected_features)
    if missing or extra:
        raise HTTPException(
            status_code=422,
            detail={"missing_features": sorted(missing),
                    "extra_features": sorted(extra)}
        )
    try:
        row = pd.DataFrame(
            [[request.features[name] for name in expected_features]],
            columns=expected_features
        )
        return PredictionResponse(
            prediction=int(model.predict(row)[0]),
            probability_class_1=float(model.predict_proba(row)[0, 1]),
            model_version=MODEL_VERSION
        )
    except Exception:
        logger.exception("Prediction failed")
        raise HTTPException(status_code=500, detail="Prediction failed")


Overwriting app.py


## 8. Run and test the API

From the project directory, after the notebook has created the artifact:

```bash
pip install fastapi uvicorn pandas scikit-learn joblib
uvicorn app:app --host 0.0.0.0 --port 8000
```

Test `GET http://localhost:8000/health` and `GET http://localhost:8000/ready`. Interactive API docs are typically available at `/docs`.

`0.0.0.0` binds to network interfaces. Protect the service with appropriate network controls; do not expose an unauthenticated model API publicly.

In [20]:
# Build a sample JSON request from one row.
sample_payload = {
    "features": {name: float(value)
                 for name, value in X_test.iloc[0].to_dict().items()}
}
print(json.dumps(sample_payload, indent=2))

{
  "features": {
    "mean radius": 19.55,
    "mean texture": 28.77,
    "mean perimeter": 133.6,
    "mean area": 1207.0,
    "mean smoothness": 0.0926,
    "mean compactness": 0.2063,
    "mean concavity": 0.1784,
    "mean concave points": 0.1144,
    "mean symmetry": 0.1893,
    "mean fractal dimension": 0.06232,
    "radius error": 0.8426,
    "texture error": 1.199,
    "perimeter error": 7.158,
    "area error": 106.4,
    "smoothness error": 0.006356,
    "compactness error": 0.04765,
    "concavity error": 0.03863,
    "concave points error": 0.01519,
    "symmetry error": 0.01936,
    "fractal dimension error": 0.005252,
    "worst radius": 25.05,
    "worst texture": 36.27,
    "worst perimeter": 178.6,
    "worst area": 1926.0,
    "worst smoothness": 0.1281,
    "worst compactness": 0.5329,
    "worst concavity": 0.4251,
    "worst concave points": 0.1941,
    "worst symmetry": 0.2818,
    "worst fractal dimension": 0.1005
  }
}


### Test the API in-process (no server needed)

FastAPI's `TestClient` (requires `pip install httpx`) calls the app directly, which makes it handy for notebooks and unit tests. Below: a valid request, then a request with a missing feature that should return 422.

In [21]:
import importlib
from fastapi.testclient import TestClient

import app as serving_app
importlib.reload(serving_app)  # pick up edits to app.py on re-run
client = TestClient(serving_app.app)

print("GET /health ->", client.get("/health").json())
print("GET /ready  ->", client.get("/ready").json())

resp = client.post("/predict", json=sample_payload)
print("POST /predict ->", resp.status_code, resp.json())

bad_payload = {"features": dict(list(sample_payload["features"].items())[:-1])}
resp = client.post("/predict", json=bad_payload)
print("POST /predict (missing feature) ->", resp.status_code, resp.json())

ImportError: cannot import name 'sentinel' from 'typing_extensions' (/Users/vivek-18890/Python/.venv310/lib/python3.10/site-packages/typing_extensions.py)

## 9. API design checklist

- Validate required fields, types, ranges and payload size.
- Keep request/response schemas stable and documented.
- Return useful status codes without leaking stack traces.
- Add authentication/authorization where required, rate limits and timeouts.
- Include model version and request/trace ID in operational metadata.
- Avoid logging sensitive raw inputs by default.

## 10. Liveness vs readiness

- **Liveness:** is the process alive and responsive? A failed liveness check may trigger restart.
- **Readiness:** can this instance currently serve traffic? It may be alive but not ready if the model has not loaded or a dependency is unavailable.

They are related but not interchangeable.

## 11. Logging and monitoring

Monitor:
- Request rate and error rate
- p50/p95/p99 latency
- CPU and memory
- Queue depth for async jobs
- Input feature distributions and prediction distributions
- Delayed ground-truth model quality

Drift is a signal to investigate, not proof by itself that model quality has declined. Labels or outcome metrics are needed to measure predictive performance.

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ml-inference-demo")

def log_prediction(model_version, latency_ms, success):
    logger.info(json.dumps({
        "event": "prediction",
        "model_version": model_version,
        "latency_ms": round(latency_ms, 3),
        "success": bool(success)
    }))

start = time.perf_counter()
success = True
try:
    loaded_model.predict(X_test.iloc[[0]])
except Exception:
    success = False
    raise
finally:
    log_prediction("1.0.0", (time.perf_counter() - start) * 1000, success)

INFO:ml-inference-demo:{"event": "prediction", "model_version": "1.0.0", "latency_ms": 0.768, "success": true}


## 12. Model versioning and metadata

Track more than a filename:
- Model name/version
- Dataset and feature schema version
- Training code commit and library versions
- Training timestamp
- Evaluation metrics
- Approval/deployment status

Keep artifacts immutable where possible so you can reproduce a release and roll back.

In [ ]:
metadata = {
    "model_name": "breast_cancer_classifier",
    "model_version": "1.0.0",
    "artifact": model_path.name,
    "features": expected_features,
    "test_accuracy": float(accuracy_score(y_test, pred)),
    "test_roc_auc": float(roc_auc_score(y_test, prob)),
    "random_state": RANDOM_STATE
}
metadata_path = ARTIFACT_DIR / "model_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Metadata:", metadata_path.resolve())

Metadata: /Users/vivek-18890/Python/learn_python/ml/13-ml-fundamentals/artifacts/model_metadata.json


## 13. Model registry — concept

A registry tracks artifacts and their lifecycle:

```text
Experiment → Registered model → Validation/approval → Staging → Production → Retired
```

It helps teams identify which model is approved, deployed and tied to an experiment. Understand the lifecycle; the exact tool API is secondary.

## 14. CPU vs GPU inference

**CPU** is often appropriate for small tabular models and modest traffic. **GPU** can help large neural networks, LLMs and batched tensor operations.

GPU is not automatically faster. Consider batch size, memory, utilization, transfer overhead, concurrency, latency and cost.

## 15. ONNX — optional conversion example

Install if supported by your model and environment: `pip install skl2onnx onnxruntime`.

Conversion compatibility varies by estimator, pipeline operators, dtypes and versions. Validate output parity and numerical tolerance before production use.

In [ ]:
# OPTIONAL — uncomment after installing skl2onnx and onnxruntime.
#
# from skl2onnx import to_onnx
# import onnxruntime as ort
#
# onnx_model = to_onnx(
#     model, X_train.iloc[:1].to_numpy(dtype=np.float32),
#     target_opset=17
# )
# onnx_path = ARTIFACT_DIR / "model.onnx"
# onnx_path.write_bytes(onnx_model.SerializeToString())
#
# session = ort.InferenceSession(
#     str(onnx_path), providers=["CPUExecutionProvider"]
# )
# input_name = session.get_inputs()[0].name
# outputs = session.run(
#     None, {input_name: X_test.iloc[:5].to_numpy(dtype=np.float32)}
# )
# print("ONNX inference executed.")
#
# Note: this example uses arrays; feature-name/schema handling and
# output interpretation must be tested for your particular model.

## 16. Dockerize the API

Example `Dockerfile`:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY artifacts/ ./artifacts/
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

Example `requirements.txt`:

```text
fastapi
uvicorn[standard]
pandas
scikit-learn
joblib
```

Build and run:

```bash
docker build -t ml-inference:1.0 .
docker run --rm -p 8000:8000 ml-inference:1.0
```

For production, pin dependency versions, scan images, avoid embedding secrets and consider fetching approved artifacts from controlled storage.

## 17. Stateless serving and horizontal scaling

```text
Clients → Load Balancer → API replicas → Model inference
```

Stateless replicas are easier to scale and replace. But each process may load its own model copy, so memory can grow with worker/replica count. Large models may need dedicated serving infrastructure. Scale based on latency, concurrency, resource use and queue depth.

## 18. Asynchronous / queue-based inference

```text
Client → API accepts job → Queue → Worker predicts → Result store → Client retrieves result
```

Useful for bulk scoring, long-running document processing and expensive inference. A queue adds operational complexity; simple, fast predictions are often better served synchronously.

## 19. Canary vs shadow deployment

**Canary:** route a small portion of real traffic to the new model, monitor it, then gradually increase traffic if healthy.

**Shadow:** copy production requests to the new model but do not use its predictions for user-facing decisions. Compare outputs, latency, errors and resource use without changing live behavior.

## 20. Rollback strategy

```text
Model v1 serving → deploy v2 → monitor
                         ├─ healthy → promote v2
                         └─ unhealthy → roll back to v1
```

Keep previous approved artifacts available and make deployment changes reversible.

## 21. Common deployment failures

1. Training and serving preprocessing differ.
2. Required features are missing or have incompatible types.
3. Model/library versions are incompatible.
4. Artifact path or permissions are wrong.
5. Model is loaded on every request.
6. Too many workers duplicate a large model in memory.
7. No timeout, overload protection or rate limit.
8. Sensitive input is logged.
9. No model version, monitoring or rollback plan.
10. Production data differs from training data.
11. CPU/GPU provider mismatch.
12. New model quality is not validated before rollout.

## 22. End-to-end deployment architecture

```text
Offline training pipeline
        ↓
Evaluation + approval
        ↓
Versioned artifact / registry
        ↓
Build serving image
        ↓
Deploy API replicas
        ↓
Load model once per process
        ↓
Validate request → preprocess → predict → response
        ↓
Log latency/errors/model version
        ↓
Monitor drift + delayed labels
        ↓
Retrain → validate → deploy → rollback if needed
```

## 23. Interview questions

**Why save the complete pipeline?** To preserve preprocessing behavior between training and inference.

**Batch vs online inference?** Online returns an immediate prediction; batch scores many records in a job.

**Why load the model at startup?** Avoid disk I/O and deserialization on every request.

**What is p95 latency?** 95% of requests complete at or below that latency.

**Liveness vs readiness?** Process alive vs able to serve traffic.

**Why version models?** Reproducibility, auditing and safe rollback.

**Why ONNX?** Portable representation/runtime for supported models.

**Why can more workers increase memory?** Processes may each hold a separate model copy.

**Canary vs shadow?** Canary serves some real traffic; shadow evaluates copied traffic without using outputs for live decisions.

**What should be monitored?** Traffic, errors, tail latency, resources, drift and eventual model quality.

## 24. 30-second interview answer

> I train and validate the model offline, then package preprocessing and the estimator as a versioned artifact. The serving application loads it once at startup, validates incoming requests, runs inference and returns a defined response. I monitor latency, error rate, resource usage, input/prediction drift and eventual model quality. For releases, I use canary or shadow testing where appropriate, retain prior approved versions and maintain a rollback path.

## 25. Final checklist

- [ ] Training vs inference
- [ ] Serialization and trusted artifacts
- [ ] Preprocessing + model pipeline
- [ ] Batch vs online inference
- [ ] REST API and input validation
- [ ] Model loaded at startup
- [ ] Liveness/readiness and error handling
- [ ] Latency, throughput and tail latency
- [ ] Logging and monitoring
- [ ] Artifact versioning / registry concepts
- [ ] CPU vs GPU trade-offs
- [ ] ONNX concept
- [ ] Docker basics
- [ ] Stateless horizontal scaling
- [ ] Queue-based inference
- [ ] Canary vs shadow deployment
- [ ] Rollback
- [ ] Training-serving skew and drift

**For your target roles:** your backend experience transfers well to APIs, containers, scaling and reliability. The ML-specific layer is artifact lifecycle, preprocessing consistency, inference behavior, model monitoring and validation.